In [22]:
import re
import json
from collections import defaultdict, Counter

# ---- Parsing ----
def parse_survey_block(block: str) -> dict:
    """Parse one <details>…</details> survey response into a dict."""
    m = re.search(r"<code>(.*?)</code>", block, re.S)
    if not m:
        return {}
    content = m.group(1)

    data = {}
    current_key = None

    for raw in content.splitlines():
        line = raw.strip()
        if not line:
            continue

        # key: value lines (ignore plugin lines which start with $:/)
        if ":" in line and not line.startswith("$:/"):
            key, value = line.split(":", 1)
            key = key.strip()
            value = value.strip()
            data[key] = value if value else None
            current_key = key
            continue

        # list items for keys like "Plugins:" (lines begin with $:/)
        if line.startswith("$:/") and current_key:
            if not isinstance(data[current_key], list):
                data[current_key] = []
            data[current_key].append(line)

    return data

# ---- Summaries ----
MULTI_VALUE_KEYS = {
    "Keyboard shortcuts that have been customised",
    "Disabled plugins",
}

def _expand_multi_values(value):
    """Keep the same 'working script' behavior: split comma-separated only."""
    if not value:
        return []
    if isinstance(value, list):
        items = []
        for v in value:
            items.extend([s.strip() for s in v.split(",") if s.strip()])
        return items
    if isinstance(value, str):
        return [s.strip() for s in value.split(",") if s.strip()]
    return []

def summarize_responses(parsed_responses):
    """Return {key: [(name, count), ...]} with counts descending."""
    counters = defaultdict(Counter)

    for resp in parsed_responses:
        for key, value in resp.items():
            if key in MULTI_VALUE_KEYS:
                counters[key].update(_expand_multi_values(value))
            elif isinstance(value, list):
                counters[key].update(value)
            else:
                counters[key].update([value if value else ""])

    # Sort by count desc, then name asc for stable output
    return {
        key: sorted(counter.items(), key=lambda kv: (-kv[1], kv[0]))
        for key, counter in counters.items()
    }

def summarize_plugins_no_version(plugin_summary_pairs):
    """Input: list[(plugin, count)], Output: list[(base_plugin, total_count)]"""
    agg = Counter()
    for plugin, count in plugin_summary_pairs:
        base = plugin.split(" - ")[0].strip()
        agg[base] += count
    return sorted(agg.items(), key=lambda kv: (-kv[1], kv[0]))

# ---- TiddlyWiki table text ----
def make_tiddlywiki_table(title, data_pairs):
    """
    data_pairs: list of (name, count)
    Returns TiddlyWiki table text with header.
    """
    lines = ["|caption-top thead-secondary sortable|k"]
    lines.append(f"|!{title} | !Count|h")
    for name, count in data_pairs:
        lines.append(f"|{name} | {count}|")
    return "\n".join(lines)

# ---- ECharts helpers ----
BAR_KEYS = {"Plugins", "Plugins (No Version)", "Disabled plugins", "Keyboard shortcuts that have been customised"}

def _make_echarts_pie_series_data(data_pairs):
    """[{value:int, name:str}, ...] from list[(name,count)]."""
    return [{"value": c, "name": n} for n, c in data_pairs]

def _make_echarts_bar_dataset(key, data_pairs):
    """
    2D dataset with header row. Header depends on key.
    - Plugins & Plugins (No Version): ["Installed Plugins","Count"]
    - Disabled plugins: ["Disabled Plugins","Count"]
    """
    if key == "Disabled plugins":
        header = ["Disabled Plugins", "Count"]
    elif key == "Keyboard shortcuts that have been customised":
        header = ["Keyboard shortcuts", "Count"]
    else:
        header = ["Installed Plugins", "Count"]
    rows = [[n, c] for n, c in data_pairs]
    return [header] + rows, header[0]  # dataset, y-encode label

def _build_pie_option(title, series_data):
    return {
        "title": {"text": title, "subtext": "TW5 Survey Response via Wiki Info Tiddler", "left": "center"},
        "tooltip": {"trigger": "item"},
        "legend": {"type": "scroll", "orient": "vertical", "left": "left"},
        "series": [{
            "name": title,
            "type": "pie",
            "radius": "50%",
            "data": series_data,
            "emphasis": {
                "itemStyle": {"shadowBlur": 10, "shadowOffsetX": 0, "shadowColor": "rgba(0, 0, 0, 0.5)"}
            },
            "label": { "show": false, "position": "center"
            },      
        }]
    }

def _build_bar_option(title, dataset, y_label):
    return {
        "title": {
            "text": title,
            "subtext": "TW5 Survey Response via Wiki Info Tiddler\n\n☑️ is default setting"
        },
        "tooltip": {"trigger": "axis", "axisPointer": {"type": "shadow"}},
        "dataZoom": [{"type": "slider", "yAxisIndex": 0, "filterMode": "none"}],
        "dataset": {"source": dataset},
        "grid": {"containLabel": True},
        "xAxis": {"name": "Count"},
        "yAxis": {"type": "category"},
        "visualMap": {
            "orient": "horizontal",
            "left": "center",
            "min": 0,
            "max": 100,
            "text": ["High Score", "Low Score"],
            "dimension": 0,
            "inRange": {"color": ["#FD665F", "#FFCE34", "#65B581"]}
        },
        "series": [{"type": "bar", "encode": {"x": "Count", "y": y_label}}]
    }

def save_multitid_json(summary, filename):
    tiddlers = []
    for key, data in summary.items():
        # Table tiddler
        tiddlers.append({
            "title": f"{key}_table",
            "type": "text/vnd.tiddlywiki",
            "text": make_tiddlywiki_table(key, data)
        })

        # Normalize to iterable of (name, count)
        items = data.items() if isinstance(data, dict) else data

        # Build ECharts option (bar for BAR_KEYS, else pie)
        chart_title = f"Tiddlywiki {key}"
        if key in BAR_KEYS:
            dataset, y_label = _make_echarts_bar_dataset(key, items)
            option = _build_bar_option(chart_title, dataset, y_label)
            is_bar = True
        else:
            series_data = _make_echarts_pie_series_data(items)
            option = _build_pie_option(chart_title, series_data)
            is_bar = False

        # _chart_data tiddler with full JS option
        chart_data_title = f"{key}_chart_data"
        tiddlers.append({
            "title": chart_data_title,
            "type": "application/javascript",
            "text": "option = " + json.dumps(option, ensure_ascii=False, indent=2) + ";"
        })

        # _chart tiddler referencing the _chart_data
        if is_bar:
            chart_text = "<$echarts $text={{" + chart_data_title + "}} $height=\"500px\"/>"
        else:
            chart_text = "<$echarts $text={{" + chart_data_title + "}}/>"

        tiddlers.append({
            "title": f"{key}_chart",
            "type": "text/vnd.tiddlywiki",
            "text": chart_text
        })

        # Special extra: Plugins (No Version)
        if key == "Plugins":
            no_ver_pairs = summarize_plugins_no_version(items)
            # table
            tiddlers.append({
                "title": "Plugins (No Version)_table",
                "type": "text/vnd.tiddlywiki",
                "text": make_tiddlywiki_table("Plugins (No Version)", no_ver_pairs)
            })
            # bar chart option
            nv_title = "Tiddlywiki Plugins (No Version)"
            nv_dataset, nv_y = _make_echarts_bar_dataset("Plugins (No Version)", no_ver_pairs)
            nv_option = _build_bar_option(nv_title, nv_dataset, nv_y)

            nv_chart_data_title = "Plugins (No Version)_chart_data"
            tiddlers.append({
                "title": nv_chart_data_title,
                "type": "application/javascript",
                "text": "option = " + json.dumps(nv_option, ensure_ascii=False, indent=2) + ";"
            })
            tiddlers.append({
                "title": "Plugins (No Version)_chart",
                "type": "text/vnd.tiddlywiki",
                "text": "<$echarts $text={{" + nv_chart_data_title + "}} $height=\"500px\"/>"
            })

    with open(filename, "w", encoding="utf-8") as f:
        json.dump(tiddlers, f, ensure_ascii=False, indent=2)


def main():
    # Load survey data
    with open("survey_responses.txt", encoding="utf-8") as f:
        raw_data = f.read()

    # Parse each <details> block
    blocks = re.findall(r"<details>.*?</details>", raw_data, re.S)
    parsed_responses = [parse_survey_block(b) for b in blocks]

    # Summarize counts for each key
    summary = summarize_responses(parsed_responses)

    # Generate tables for each key (print to console)
    for key, data in summary.items():
        print(f"\n---\n{make_tiddlywiki_table(key, data)}\n")

        # Special extra table for Plugins (no version)
        if key == "Plugins":
            no_ver = summarize_plugins_no_version(data)
            print(make_tiddlywiki_table("Plugins (No Version)", no_ver))
            print()

    # Also produce multi‑tid files (tables + charts), including Plugins (No Version)
    extended_summary = dict(summary)
    if "Plugins" in summary:
        extended_summary["Plugins (No Version)"] = summarize_plugins_no_version(summary["Plugins"])

    #save_tables_multitid(extended_summary, "survey_summary_tables_multitid.json")
    #save_charts_multitid(extended_summary, "survey_summary_charts_multitid.json")
    save_multitid_json(summary, "survey_summary.json")

    print("Survey summary generated.")

if __name__ == "__main__":
    main()




---
|caption-top thead-secondary sortable|k
|!TiddlyWiki Version | !Count|h
|5.3.6 | 43|
|5.3.7 | 39|
|5.3.3 | 11|
|5.3.5 | 11|
|5.3.1 | 4|
|5.3.2 | 3|
|5.3.8 | 2|
|5.3.0 | 1|


---
|caption-top thead-secondary sortable|k
|!Current palette | !Count|h
|$:/palettes/Vanilla | 45|
|$:/palettes/Twilight | 8|
|$:/palettes/CupertinoDark | 6|
|$:/palettes/DesertSand | 4|
|$:/palettes/Nord | 4|
|$:/palettes/Blue | 2|
|$:/palettes/Dracula | 2|
|$:/palettes/FlexokiLight | 2|
|$:/palettes/GithubDark | 2|
|$:/palettes/Muted | 2|
|$:/palettes/SpartanDay | 2|
|$:/.oap/palettes/NordOAP | 1|
|$:/JMW/palettes/Nightfall | 1|
|$:/palettes/ABL-COLORS | 1|
|$:/palettes/Blanca | 1|
|$:/palettes/Blanca_wes | 1|
|$:/palettes/CaptivateDark (MY OWN TWEAKED PALETTE) | 1|
|$:/palettes/CaptivateTan | 1|
|$:/palettes/ContrastDark | 1|
|$:/palettes/DarkTheme | 1|
|$:/palettes/Eberhard | 1|
|$:/palettes/EzVanilla | 1|
|$:/palettes/FlexokiDark | 1|
|$:/palettes/FlexokiDark EDITED 1 | 1|
|$:/palettes/Fresh | 1|
|$:/pal

NameError: name 'false' is not defined